In [4]:
import numpy as np
import pandas as pd
import matplotlib 
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [21]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

In [6]:
#Load dataset
df = pd.read_csv("novagen_dataset.")

In [8]:
# boolean columns into integer
bool_cols = [
    'Diet_Type__Vegan', 'Diet_Type__Vegetarian',
    'Blood_Group_AB', 'Blood_Group_B', 'Blood_Group_O'
]
for col in bool_cols:
    df[col] = df[col].astype(int)

print(f"Records       : {df.shape[0]}")
print(f"Features      : {df.shape[1] - 1}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"\nTarget distribution:\n{df['Target'].value_counts().to_string()}")

Records       : 9549
Features      : 22
Missing values: 0

Target distribution:
Target
1    4979
0    4570


In [9]:
# data analysis
n=len(df)
c={'healthy': '#2ecc71', 'unhealthy': '#e74c3c'}
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    'Exploratory Data Analysis - NovaGen Health Dataset',
    fontsize=15, fontweight='bold'
)

Text(0.5, 0.98, 'Exploratory Data Analysis - NovaGen Health Dataset')

In [10]:
# 2a. target distribution
ax = axes[0,0]
counts = df['Target'].value_counts().sort_index()
bars = ax.bar(
    ['Healthy (0)', 'Unhealthy (1)'], counts.values,
    color=[c['healthy'], c['unhealthy']], edgecolor='white', linewidth=1.5
)
for bar, val in zip(bars, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 30,
        f'{val}\n({val / n * 100:.1f}%)',
        ha='center', va='bottom', fontweight='bold', fontsize=9
    )
ax.set_title('Target Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
ax.set_ylim(0, max(counts.values) * 1.2)

(0.0, 5974.8)

In [11]:
# 2b. BMI distribution by class
ax=axes[0,1]
for label,color,name in [(0, c['healthy'], 'Healthy'), (1, c['unhealthy'], 'Unhealthy')]:
    ax.hist(df[df['Target'] == label]['BMI'].dropna(), bins=25, alpha=0.65,
            color=color, label=name)
ax.set_title('BMI Distribution by Class', fontweight='bold')
ax.set_xlabel('BMI')
ax.set_ylabel('Frequency')
ax.legend()


In [12]:
# 2c. Age distribution by class
ax = axes[0, 2]
for label, color, name in [(0, c['healthy'], 'Healthy'), (1, c['unhealthy'], 'Unhealthy')]:
    ax.hist(df[df['Target'] == label]['Age'].dropna(), bins=25, alpha=0.65,
            color=color, label=name)
ax.set_title('Age Distribution by Class', fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Frequency')
ax.legend()

In [13]:
# 2d. Correlation heatmap
ax = axes[1, 0]
num_cols = [
    'Age', 'BMI', 'Blood_Pressure', 'Cholesterol',
    'Glucose_Level', 'Stress_Level', 'Exercise_Hours', 'Sleep_Hours', 'Target'
]
corr = df[num_cols].corr()
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
    ax=ax, cbar=True, annot_kws={'size': 7}
)
ax.set_title('Feature Correlation Matrix', fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=7)

In [14]:
# 2e. Blood Pressure vs Glucose
ax = axes[1, 1]
for label, color, name in [(0, c['healthy'], 'Healthy'), (1, c['unhealthy'], 'Unhealthy')]:
    s = df[df['Target'] == label]
    ax.scatter(s['Blood_Pressure'], s['Glucose_Level'],
               alpha=0.2, s=7, color=color, label=name)
ax.set_title('Blood Pressure vs Glucose Level', fontweight='bold')
ax.set_xlabel('Blood Pressure (mmHg)')
ax.set_ylabel('Glucose Level (mg/dL)')
ax.legend()

In [15]:
# 2f. Unhealthy rate by lifestyle factor
ax = axes[1, 2]
cats = ['Non-Smoker', 'Smoker', 'No Alcohol', 'Alcohol']
pcts = [
    df[df['Smoking'] == 0]['Target'].mean() * 100,
    df[df['Smoking'] == 1]['Target'].mean() * 100,
    df[df['Alcohol'] == 0]['Target'].mean() * 100,
    df[df['Alcohol'] == 1]['Target'].mean() * 100,
]
bar_cols = [c['healthy'], c['unhealthy'], c['healthy'], c['unhealthy']]
bars = ax.bar(cats, pcts, color=bar_cols, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Unhealthy Rate by Lifestyle Factor', fontweight='bold')
ax.set_ylabel('% Classified as Unhealthy')
ax.tick_params(axis='x', rotation=10)
 
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.close()
print("EDA chart saved: eda_overview.png")

EDA chart saved: eda_overview.png


In [16]:
# preprocessing
X = df.drop('Target', axis=1)
y = df['Target']
 
# No missing values in this dataset - confirmed during load
# Train / test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
# Feature scaling
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)
 
print(f"Train samples : {len(X_train)}")
print(f"Test samples  : {len(X_test)}")
print(f"Train class distribution: {dict(y_train.value_counts())}")

Train samples : 7639
Test samples  : 1910
Train class distribution: {1: np.int64(3983), 0: np.int64(3656)}


In [22]:
# Model Training
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=7),
    'SVM':                 SVC(probability=True, random_state=42),
}
 
results = {}
for name, model in models.items():
    model.fit(X_tr, y_train)
    yp   = model.predict(X_te)
    ypr  = model.predict_proba(X_te)[:, 1]
    cv   = cross_val_score(model, X_tr, y_train, cv=5, scoring='accuracy').mean()
    results[name] = {
        'Accuracy':  accuracy_score(y_test, yp),
        'F1':        f1_score(y_test, yp),
        'Precision': precision_score(y_test, yp),
        'Recall':    recall_score(y_test, yp),
        'AUC':       roc_auc_score(y_test, ypr),
        'CV':        cv,
        'yp':        yp.tolist(),
        'ypr':       ypr.tolist(),
    }
    print(
        f"{name:25s}  Acc: {results[name]['Accuracy']:.4f}  "
        f"F1: {results[name]['F1']:.4f}  "
        f"AUC: {results[name]['AUC']:.4f}  "
        f"CV: {cv:.4f}"
    )

Logistic Regression        Acc: 0.8136  F1: 0.8224  AUC: 0.8879  CV: 0.8167
Decision Tree              Acc: 0.8597  F1: 0.8665  AUC: 0.9229  CV: 0.8656
Random Forest              Acc: 0.9366  F1: 0.9402  AUC: 0.9845  CV: 0.9340
Gradient Boosting          Acc: 0.9199  F1: 0.9248  AUC: 0.9721  CV: 0.9184


  File "C:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\ProgramData\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\subprocess.

KNN                        Acc: 0.8901  F1: 0.8947  AUC: 0.9485  CV: 0.8732
SVM                        Acc: 0.9335  F1: 0.9371  AUC: 0.9776  CV: 0.9296


In [23]:
# model comparison plots
names = list(results.keys())
accs  = [results[n]['Accuracy']  for n in names]
f1s   = [results[n]['F1']        for n in names]
aucs  = [results[n]['AUC']       for n in names]
cvs   = [results[n]['CV']        for n in names]
 
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')
 
ax = axes[0]
x = np.arange(len(names)); w = 0.2
ax.bar(x,       accs, w, label='Accuracy',    color='#3498db', alpha=0.85)
ax.bar(x + w,   f1s,  w, label='F1 Score',    color='#e74c3c', alpha=0.85)
ax.bar(x + 2*w, aucs, w, label='AUC-ROC',     color='#2ecc71', alpha=0.85)
ax.bar(x + 3*w, cvs,  w, label='CV Accuracy', color='#f39c12', alpha=0.85)
ax.set_xticks(x + 1.5*w)
ax.set_xticklabels(names, rotation=20, ha='right', fontsize=8)
ax.set_ylim(0.5, 1.05)
ax.set_title('Metrics by Model', fontweight='bold')
ax.legend(fontsize=8)
ax.set_ylabel('Score')
 
ax = axes[1]
roc_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
for (nm, res), col in zip(results.items(), roc_colors):
    fpr, tpr, _ = roc_curve(y_test, res['ypr'])
    ax.plot(fpr, tpr, label=f"{nm} ({res['AUC']:.3f})", color=col, lw=1.8)
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_title('ROC Curves', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=7)
 
ax = axes[2]
cv_pairs = sorted(zip(cvs, names))
sv = [v for v, n in cv_pairs]
sn = [n for v, n in cv_pairs]
max_cv = max(cvs)
bar_colors = ['#2ecc71' if v == max_cv else '#3498db' for v in sv]
bars = ax.barh(sn, sv, color=bar_colors)
for bar, val in zip(bars, sv):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_xlim(0.5, 1.02)
ax.set_title('5-Fold Cross-Validation Accuracy', fontweight='bold')
ax.set_xlabel('CV Accuracy')
 
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("Model comparison chart saved: model_comparison.png")

Model comparison chart saved: model_comparison.png


In [24]:
# hyperparameter tuning - random forest (best cv)
param_grid = {
    'n_estimators':     [100, 200],
    'max_depth':        [None, 10, 20],
    'min_samples_split':[2, 5],
}
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid, cv=3, scoring='f1', n_jobs=-1
)
rf_grid.fit(X_tr, y_train)
best_rf = rf_grid.best_estimator_
 
print(f"Best parameters : {rf_grid.best_params_}")
 
yp_best  = best_rf.predict(X_te)
ypr_best = best_rf.predict_proba(X_te)[:, 1]
 
print(f"\nTuned Random Forest Results:")
print(f"  Accuracy  : {accuracy_score(y_test, yp_best):.4f}")
print(f"  F1 Score  : {f1_score(y_test, yp_best):.4f}")
print(f"  AUC-ROC   : {roc_auc_score(y_test, ypr_best):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, yp_best, target_names=['Healthy', 'Unhealthy']))

Best parameters : {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}

Tuned Random Forest Results:
  Accuracy  : 0.9387
  F1 Score  : 0.9424
  AUC-ROC   : 0.9844

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.96      0.91      0.93       914
   Unhealthy       0.92      0.96      0.94       996

    accuracy                           0.94      1910
   macro avg       0.94      0.94      0.94      1910
weighted avg       0.94      0.94      0.94      1910



In [25]:
# feature importance and confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(
    'Best Model Analysis - Random Forest (Tuned)',
    fontsize=14, fontweight='bold'
)
 
ax = axes[0]
fi = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=True)
ci = ['#e74c3c' if v >= fi.quantile(0.75) else '#3498db' for v in fi]
fi.plot(kind='barh', ax=ax, color=ci)
ax.set_title('Feature Importances', fontweight='bold')
ax.set_xlabel('Importance Score')
ax.legend(
    handles=[
        mpatches.Patch(color='#e74c3c', label='Top 25% Features'),
        mpatches.Patch(color='#3498db', label='Remaining Features')
    ],
    fontsize=9
)
 
ax = axes[1]
cm = confusion_matrix(y_test, yp_best)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['Healthy', 'Unhealthy'],
    yticklabels=['Healthy', 'Unhealthy'],
    linewidths=1, linecolor='white', cbar=False,
    annot_kws={'size': 14, 'weight': 'bold'}
)
ax.set_title('Confusion Matrix', fontweight='bold')
ax.set_xlabel('Predicted Label', fontweight='bold')
ax.set_ylabel('True Label', fontweight='bold')
tn, fp, fn, tp = cm.ravel()
for (i, j, lbl) in [
    (0, 0, f'TN={tn}'), (0, 1, f'FP={fp}'),
    (1, 0, f'FN={fn}'), (1, 1, f'TP={tp}')
]:
    ax.text(j + 0.5, i + 0.75, lbl, ha='center', va='center', color='gray', fontsize=10)
 
plt.tight_layout()
plt.savefig('best_model_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("Best model analysis chart saved: best_model_analysis.png")


Best model analysis chart saved: best_model_analysis.png


In [26]:
summary_df = pd.DataFrame(
    {n: {m: v for m, v in res.items() if m not in ('yp', 'ypr')}
     for n, res in results.items()}
).T.round(4)
 
print("\nAll Models - Performance Summary:")
print(summary_df[['Accuracy', 'F1', 'Precision', 'Recall', 'AUC', 'CV']].to_string())
 
best_by_auc = summary_df['AUC'].idxmax()
print(f"\nBest Model (by AUC-ROC) : {best_by_auc}")
print(f"  AUC-ROC  : {summary_df.loc[best_by_auc, 'AUC']:.4f}")
print(f"  Accuracy : {summary_df.loc[best_by_auc, 'Accuracy']:.4f}")
print(f"  F1 Score : {summary_df.loc[best_by_auc, 'F1']:.4f}")
 
print("\nAll outputs saved successfully.")


All Models - Performance Summary:
                     Accuracy      F1  Precision  Recall     AUC      CV
Logistic Regression    0.8136  0.8224     0.8175  0.8273  0.8879  0.8167
Decision Tree          0.8597  0.8665     0.8597  0.8735  0.9229  0.8656
Random Forest          0.9366  0.9402     0.9260  0.9548  0.9845  0.9340
Gradient Boosting      0.9199  0.9248     0.9057  0.9448  0.9721  0.9184
KNN                    0.8901  0.8947     0.8938  0.8956  0.9485  0.8732
SVM                    0.9335  0.9371     0.9247  0.9498  0.9776  0.9296

Best Model (by AUC-ROC) : Random Forest
  AUC-ROC  : 0.9845
  Accuracy : 0.9366
  F1 Score : 0.9402

All outputs saved successfully.
